# CAS Exam 5: Chain Ladder Method (Development Method)

**Source:** Friedland, J. *Estimating Unpaid Claims Using Basic Techniques*, Casualty Actuarial Society, 2010

This notebook uses real data from:
- `.../chainladder-python/chainladder/utils/data/friedland_us_industry_auto.csv`

Learning goals:
- Work directly with the `chainladder.Triangle` structure.
- Show age-to-age factors, candidate LDF selections, tail alternatives, and selected CDFs.
- Project ultimates and implied IBNR with explicit actuarial considerations.

## Formula Sheet Reference

### 7-Step Chain Ladder Process

| Step | Action |
|------|--------|
| 1 | Compile claims data into a triangle (cumulative paid **or** reported, by origin × development age) |
| 2 | Calculate **age-to-age development factors** for each origin year and each transition |
| 3 | Calculate **averages** of age-to-age factors: volume-weighted, simple (arithmetic), medial (trimmed), geometric |
| 4 | **Select** claim development factors (LDFs) for each development period |
| 5 | **Select a tail factor** — covers development beyond the oldest observed age |
| 6 | Calculate **cumulative development factors (CDFs)** by multiplying selected LDFs from each age to ultimate |
| 7 | **Project ultimate claims** = Latest Diagonal × CDF-to-ultimate |

### Key Relationships

$$\text{CDF}(d \to \text{ult}) = \text{LDF}(d \to d{+}1) \times \text{LDF}(d{+}1 \to d{+}2) \times \cdots \times \text{Tail}$$

$$\text{Ultimate} = \text{Latest} \times \text{CDF}(d \to \text{ult})$$

$$\text{IBNR} = \text{Ultimate} - \text{Latest} = \text{Latest} \times (\text{CDF} - 1)$$

$$\%\ \text{Paid at age } d = \frac{1}{\text{CDF}(d \to \text{ult})} \qquad \%\ \text{Unreported} = 1 - \frac{1}{\text{CDF}}$$

### Averaging Methods (Step 3)

- **Volume-weighted**: $\frac{\sum C_{w,d+1}}{\sum C_{w,d}}$ — weights by the prior-age value; most common selection
- **Simple (arithmetic)**: unweighted mean of all link ratios for that transition
- **Medial**: drop the single highest and lowest link ratio, then take the simple average of the rest
- **Geometric**:  $ \exp\!\left(\frac{1}{n}\sum_{i=1}^{n}\ln f_i\right)\;=\;\left(\prod_{i=1}^{n} f_i\right)^{1/n}\ $  — less sensitive to outlier-high factors than arithmetic

### Key Assumptions

1. Future development will follow patterns consistent with observed history
2. Each accident year's observed claims at a given age contain information about its ultimate
3. The mix of business, claim handling, and external environment are sufficiently stable for historical patterns to transfer forward

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import chainladder as cl  
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
ROOT = Path.cwd().resolve()
# if we’re in the notebooks directory, step up to the repo root
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "chainladder-python" / "chainladder" / \
            "utils" / "data" / "friedland_us_industry_auto.csv"
raw = pd.read_csv(DATA_PATH)

raw.head()

*Initial Data Requirements*:
* Tabular data with:
    * one origin field (Accident Year)
    * one development (or valuation) field (Calendar Year)
    * one or more value columns (Paid Claims, Reported Claims)
* Data may be incremental or cumulative.
* Optional: one or more segment/index fields. (e.g. LOBs. Only auto data is used in this example)

## Step 1: Compile Data in `chainladder.Triangle` Form

The object below is the actual chainladder triangle structure used for all downstream calculations.    
*Structure*  
* 4D object: [index, columns, origin, development]  
    * index = segment keys (e.g., LOB, state, company); can be multi-level  
    * columns = measures (Paid, Incurred, Counts, Exposure, etc.)  
    * origin = accident/policy/report year (or period)  
    * development = development age (or valuation period)  


In [ ]:
triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Paid Claims', 'Reported Claims'],
    cumulative=True,
)
triangle

In [ ]:
paid_triangle = triangle['Paid Claims']
reported_triangle = triangle['Reported Claims']
paid_triangle

In [ ]:
paid_triangle.shift(1)

In [ ]:
print("Origin:", paid_triangle.origin) # origin axis values
print("Development Ages:", paid_triangle.development.values) # development ages
print("Valuation Date:", paid_triangle.valuation_date) # latest valuation date of the triangle
print("Columns:", paid_triangle.columns) # measure names
print("Index:", paid_triangle.index) # segment keys. Default is Total
# index and slice triangles similar to pandas, but 4d
print("Shape:", paid_triangle.shape) # (index, values, number of origin periods, number of development ages)
print("Is Cumulative:", paid_triangle.is_cumulative) # indicates cumulative vs incremental

In [ ]:
print("Latest Diagonal:") 
paid_triangle.latest_diagonal # most recent observed values

In [ ]:
paid_triangle.describe()

In [ ]:
# paid_triangle.groupby("LOB").sum()
# groupby — works on the index axis (dim 0), which requires a multi-row index. 
# #The Friedland dataset is a single-entity triangle (shape[0] = 1), 
#   so groupby does nothing useful there.

In [ ]:
paid_triangle.grain("OYDY")

In [ ]:
(paid_triangle / reported_triangle).heatmap()

In [ ]:
print("Paid Losses for every maturity of first 3 origin periods :\n")
paid_triangle.values[0, 0, :3, :] # underlying numpy array 

In [ ]:
paid_triangle.heatmap()

In [ ]:
paid_triangle.plot()

In [ ]:
paid_triangle.dev_to_val()

In [ ]:
paid_triangle.cum_to_incr()

**Reading incremental data:** Incremental claims show how much development occurred *within each calendar year* (each diagonal). Three things to look for:

- **Diagonal spikes** — unusually large incremental amounts for one calendar year that appear across multiple origin years (same column in the incremental triangle) suggest a systemwide event: superimposed inflation, a legal change, or a shift in claims handling. This is the visual signal that the L/S calendar-year test below formalises.
- **Heavy late-age increments** — if the oldest development periods still show material payments, the tail factor matters significantly and a 1.000 tail would understate ultimates.
- **Near-zero late increments** — if the last few development ages are essentially zero, the line is nearly fully developed within the triangle and a tail close to 1.000 is defensible.

## Step 2: Age-to-Age Factors and Pattern Diagnostics

We inspect link ratios by AY. Educationally, this is where we look for trend, volatility, and possible structural shifts.
If early-age factors are drifting down or up over time, that influences selection of recent vs all-year averages.


In [ ]:
age_to_age = paid_triangle.link_ratio
age_to_age.heatmap()


### Full Age-to-Age Factor Exhibit (Friedland Style — Step 2 / Step 3)

The heatmap above gives a visual read; the exhibit below gives the numbers. This is the standard Friedland presentation: every individual link ratio by accident year, with the four formula-sheet averages appended as summary rows.

> **Formula sheet note** — Exam questions can ask you to compute any of the four averaging methods from a factor table, or to identify which average a given set of numbers represents. The table below shows all four for every development period side by side.

In [ ]:
# Geometric average — not built into chainladder Development, so computed manually.
# Formula: exp(mean of log link-ratios), which is less pulled by outlier-high factors than arithmetic.

lr_frame = age_to_age.to_frame()   # individual link ratios: origins × transitions
geo_avg = pd.Series(
    np.exp(np.log(lr_frame.clip(lower=1e-8)).mean()),
    name='Geometric Avg',
    dtype=float,
)

print("Geometric averages by development period:")
geo_avg.to_frame().T

### Commentary on the 12–24 Age-to-Age Factors (Accident Years 1998–2006)

The 12–24 development factors show a **general downward trend across accident years**, with relatively small year-to-year fluctuations.

#### 1. Observed Pattern

* Earlier accident years (e.g., 1998–1999) exhibit higher 12–24 factors.
* More recent accident years show gradually smaller factors.
* The year-over-year change is modest and mostly negative.
* There are no extreme spikes or volatility.

This indicates a **smooth declining pattern** rather than random instability.

---

### 2. Interpretation of a Decreasing 12–24 Factor

The 12–24 factor reflects the amount of development occurring between the first and second year after accident.

A decreasing factor over time may suggest:

* **Faster claim reporting or settlement** in more recent years.
* Improved claims handling or operational efficiency.
* Shifts in case reserving philosophy.
* A reduction in late-reported claims.
* Structural changes in mix of business.

The key actuarial question is:

> Is this trend structural (permanent change) or random fluctuation?

---

### 3. Relevance for Development Method Assumptions

The Chain Ladder method assumes:

* Development patterns are stable over time.
* Future development will resemble historical development.

A consistent downward drift may indicate:

* Violation of strict stationarity.
* A possible calendar year or operational effect.
* A change in reporting speed.

This does **not automatically invalidate Chain Ladder**, but it requires judgment.

---

### 4. Factor Selection Considerations (Exam Perspective)

When selecting the 12–24 factor, several approaches may be considered:

#### Option A: Volume-Weighted Average

* Appropriate if trend is modest and data is credible.
* Smooths minor fluctuations.
* Most commonly used default.

#### Option B: Simple or Recent Average

* May be appropriate if operational changes suggest older years are not representative.
* More weight on recent accident years.

#### Option C: Exclude Early Accident Years

* If earlier high factors are believed to reflect outdated claims practices.
* Common exam scenario: "process change occurred in year X."

#### Option D: Trend Adjustment

* If a structural acceleration in reporting is believed permanent,
  projecting a slightly lower factor may be justified.

---

### 5. Stability and Credibility Assessment

Indicators supporting use of traditional Chain Ladder:

* Smooth progression.
* No extreme volatility.
* Changes are gradual rather than abrupt.
* No obvious outliers.

Indicators suggesting caution:

* Systematic monotonic decline.
* Known operational change during period.
* Evidence of valuation correlation.

---

### 6. Potential Exam Questions Based on This Pattern

If presented with this table, typical questions might include:

* Is the 12–24 factor stable?
* Should earlier accident years be excluded?
* Is there evidence of a change in settlement speed?
* How would faster reporting affect ultimates?
* Would the development method overstate or understate ultimates if the trend continues?
* How would this impact Bornhuetter–Ferguson?
* Would valuation correlation likely detect this?

---

### 7. Impact on Ultimate Estimates

If more recent years truly develop less between 12–24:

* Using older higher factors may **overstate ultimates**.
* Using only recent lower factors may reduce projected development.
* Choice of selected factor materially affects immature accident years.

Since 12–24 is an early maturity period, its selection significantly impacts:

* Young accident years
* IBNR estimates
* Sensitivity of ultimates

In [ ]:
first_factor = age_to_age.to_frame()['12-24'].dropna().to_frame(name='12-24')
first_factor['ChangeFromPriorAY'] = first_factor['12-24'].diff()
first_factor

This uses chainladder's Mack-style diagnostics (not exam-5):  
- `DevelopmentCorrelation` for serial/development-period correlation.
    * H_0: Independence between successive development factors (rejected)
- `ValuationCorrelation` for calendar-year/valuation effects.  

In [ ]:
development_corr = cl.DevelopmentCorrelation(paid_triangle, p_critical=0.5)
valuation_corr = cl.ValuationCorrelation(paid_triangle, p_critical=0.1, total=True)

valuation_z = float(valuation_corr.z.iloc[0, 0]) if hasattr(valuation_corr.z, "iloc") else float(valuation_corr.z)
valuation_flag = bool(valuation_corr.z_critical.iloc[0, 0]) if hasattr(valuation_corr.z_critical, "iloc") else bool(valuation_corr.z_critical)

correlation_summary = pd.DataFrame(
    [
        {
            'DevelopmentCorrReject': bool(development_corr.reject),
            'Development_p_critical': development_corr.p_critical,
            'ValuationZ': valuation_z,
            'ValuationCorrFlag': valuation_flag,
            'Valuation_p_critical': valuation_corr.p_critical,
        }
    ]
)

correlation_summary


Development Correlation Test: Reject the null
*Translation:*
* Adjacent development periods are correlated.  
* The Mack independence assumption is violated.  
* Development factors are not independent across columns.  

This affects:  
* Mack standard error estimates  
* Variance calculations  
* Confidence intervals   
* *It does not automatically invalidate point estimates, but it weakens the statistical assumptions behind the model.*  

Why this matters  
* If development factors are correlated: 
    * Variance estimates are understated.  
    * The triangle may have structural effects.  
*You should be cautious interpreting Mack results.*

The valuation correlation test does not indicate significant calendar-year effects, suggesting the core Chain Ladder assumption that development depends only on age is not materially violated.

### Calendar Year / Diagonal Effects — L/S Test (Friedland Chapter 8)

The `ValuationCorrelation` test above gives a statistical flag. The **L/S test** gives the intuition behind it.

**How it works:**
1. For each development period (column), compute the column's median link ratio.
2. Label every individual link ratio "**L**" if it is above the median, "**S**" if at or below.
3. For each calendar year (diagonal), count the L's and S's across all development periods active in that year.
4. A year with predominantly L's had above-average development across the board — possible superimposed inflation, a favourable legal year, or a claims-handling change.  A year with predominantly S's had the reverse.

A calendar year is flagged as unusual if it is consistently skewed.  This is complementary to the statistical test: the test tells you *whether* an effect exists; the table below tells you *which year(s)* to investigate.

In [ ]:
from reservingengine.reserving import calendar_year_diagnostic
cy_diag = calendar_year_diagnostic(paid_triangle)

# L/S matrix: 'L' = factor above column median, 'S' = at or below
# Read along diagonals (top-right to bottom-left) to see calendar years
print("L/S matrix — read diagonals to identify calendar year effects:")
cy_diag['ls_matrix']

In [ ]:
# Calendar year summary: count of L and S labels per diagonal year
# pct_L near 0.75+ = above-average development year; near 0.25- = below-average
print("Calendar year summary:")
cy_diag['calendar_year_summary']

### Paid vs Reported (Incurred) Development Comparison

Both triangles were loaded in Step 1 but only paid is used for the main projection. Comparing paid and reported development patterns is a standard diagnostic before committing to either:

- **Paid LDFs > Reported LDFs** at the same age: paid is developing faster than reported, which means case reserves are being paid down quickly. Paid development can be more volatile if settlement rates shift.
- **Paid LDFs < Reported LDFs**: reported grows faster than paid — case reserves are building up faster than they're being paid, possibly signaling case reserve strengthening.
- **Paid/Incurred ratio at latest diagonal** should approach **1.0** for mature accident years. For immature years, a ratio well below 1.0 is expected (significant case reserves still outstanding). A ratio *above* 1.0 is a data error.

> **Exam connection** — Berquist-Sherman adjustments are motivated by exactly this comparison. When case reserve adequacy changes over time, it distorts reported development patterns and you must adjust the triangle before applying the development method.  If case reserves increase (decrease) in adequacy, reported LDFs are artificially depressed (inflated).

In [ ]:
from reservingengine.reserving import paid_vs_incurred_comparison

pi_comp = paid_vs_incurred_comparison(paid_triangle, reported_triangle)

# Side-by-side selected LDFs + ratio column (paid LDF / reported LDF)
# Values < 1.0 mean paid develops faster than reported at that age
print("LDF comparison — paid vs reported, with ratio (paid/reported):")
pi_comp['ldf_comparison']

In [ ]:
# CDF comparison: paid vs reported CDFs to ultimate
# Paid CDF / Reported CDF > 1 means paid has more development remaining (higher leverage)
print("CDF-to-ultimate comparison — paid vs reported:")
pi_comp['cdf_comparison']

In [ ]:
# Paid/incurred ratio at the latest diagonal by accident year.
# Mature years (oldest AYs) should be near 1.0.
# Recent years are expected to be lower — large case reserves still outstanding.
# A ratio persistently below 1.0 even for mature years may suggest case reserves
# are building, which would inflate reported development factors.
print("Paid / incurred ratio at latest diagonal:")
pi_comp['paid_to_incurred_latest']

## Steps 3-4: Candidate LDF Sets and Selected Pattern

Candidate sets shown below:
- `simple_all_periods` — arithmetic (simple) average of all link ratios
- `volume_all_periods` — volume-weighted average of all years (formula-sheet standard)
- `regression_all_periods` — OLS regression through the origin
- `volume_recent_5` — volume-weighted average of the 5 most recent years

Additional averages computed separately (not built into `cl.Development`):
- **Geometric** — `exp(mean of log link-ratios)`; formula sheet lists this explicitly
- **Medial** — drop highest and lowest, then simple-average the rest; formula sheet lists this explicitly

Selection used here for teaching: recent-5 volume for the first two intervals (where trend pressure is highest), then all-year volume for later intervals (where credibility is lower and stability matters).

Note: for chainladder-python (as with sklearn), `.fit()` stores the parameters & attributes (`.ultimate_`, `.ibnr_`, `.latest_diagonal`) expose results

In [ ]:
candidate_specs = {
    'simple_all_periods': {'average': 'simple', 'n_periods': -1},
    'volume_all_periods': {'average': 'volume', 'n_periods': -1},
    'regression_all_periods': {'average': 'regression', 'n_periods': -1},
    'volume_recent_5': {'average': 'volume', 'n_periods': 5},
}

candidate_models = {name: cl.Development(**spec).fit(paid_triangle) for name, spec in candidate_specs.items()}
ldf_comparison = pd.DataFrame({name: model.ldf_.to_frame().iloc[0] for name, model in candidate_models.items()}).T

selected_average = ['volume'] * ldf_comparison.shape[1]
selected_n_periods = [5, 5] + [-1] * (ldf_comparison.shape[1] - 2)
selected_dev_model = cl.Development(average=selected_average, n_periods=selected_n_periods).fit(paid_triangle)
selected_ldf = selected_dev_model.ldf_.to_frame().iloc[0]

ldf_comparison.loc['selected_hybrid'] = selected_ldf
ldf_comparison


In [ ]:
# Extend the comparison with the two formula-sheet averages not built into cl.Development:
# geometric and medial.  These use the same link ratios already computed in Step 2.
# Geometric: exp(mean of log link-ratios)
geo_row = pd.Series(
    np.exp(np.log(lr_frame.clip(lower=1e-8)).mean()),
    dtype=float,
)


ldf_comparison_all = ldf_comparison.copy()
ldf_comparison_all.loc['geometric_all_periods'] = geo_row

print("All six candidate LDF sets including formula-sheet geometric and medial averages:")
ldf_comparison_all

Volume-weighted averages were selected due to their credibility and stability. Early maturities show modest downward drift in recent years; therefore, recent-year experience was emphasized for immature ages. Later maturities are stable across methods, so all-period volume-weighted averages were deemed appropriate. The selected factors produce a smooth progression and reflect both historical credibility and recent experience.

## Step 5: Tail Factor Choices

Three practical tail views demonstrated:
- Industry benchmark constant tail (1.010).
- Data-fitted `TailCurve` from the selected developed triangle.
- Paid-vs-reported proxy using mature AY latest reported/paid relationship.

No single tail is universally correct; appropriateness depends on line behavior, maturity, and external benchmark support.


**Short-tail vs long-tail context for tail factor selection:**

| Line type | Typical tail range | Notes |
|-----------|-------------------|-------|
| Short-tail (PD, homeowners, auto physical damage) | 1.000–1.005 | Triangle usually 95%+ developed within 3–5 years; tail is rarely material |
| Medium-tail (auto liability, GL small claims) | 1.005–1.030 | Some residual IBNR from late-reported or reopened claims |
| Long-tail (workers comp, med-mal, excess/umbrella) | 1.05–1.30+ | Tail can be the **single largest source of reserve uncertainty**; a 0.01 error in the tail factor has the same dollar impact as a 0.01 error in every selected LDF combined |

**When the tail matters most:**
- The oldest triangle age is still materially developing (less than ~85% reported at last age)
- The line has slow-settling bodily-injury or liability claims  
- Data is sparse at late ages, making curve extrapolation unreliable (TailCurve's weakness)
- External benchmarks differ materially from the data-fitted curve (suggests the data may not yet show full late-age emergence)

In [ ]:
selected_dev_triangle = selected_dev_model.transform(paid_triangle)
tail_curve_model = cl.TailCurve().fit(selected_dev_triangle) # extrapolate development beyond final maturity
tail_curve_factor = float(tail_curve_model.tail_.iloc[0, 0]) # extract tail factor from model

In [ ]:
tail_curve_factor

In [ ]:
tail_curve_model

In [ ]:
#set up matrices for reported vs paid ratio analysis at mature ages
paid_long = paid_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
reported_long = reported_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
paid_matrix = paid_long.pivot(index='origin', columns='development', values='Paid Claims').sort_index().sort_index(axis=1)
reported_matrix = reported_long.pivot(index='origin', columns='development', values='Reported Claims').sort_index().sort_index(axis=1)

latest_age = paid_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
mature_mask = latest_age >= 96

# get reported/paid ratio at mature ages 
mature_ratio = (reported_matrix.max(axis=1) / paid_matrix.max(axis=1)).where(mature_mask)
reported_paid_proxy_tail = float(mature_ratio.mean())

tail_candidates = pd.DataFrame(
    {
        'TailFactor': {
            'NoTail_1.000': 1.000,
            'IndustryBenchmark_example': 1.010,
            'TailCurve_FromData': tail_curve_factor,
            'MatureReportedPaidProxy': reported_paid_proxy_tail,
        },
        'Consideration': {
            'NoTail_1.000': 'Only reasonable if practically fully developed by 120 months.',
            'IndustryBenchmark_example': 'External benchmark; useful when data is sparse or noisy.',
            'TailCurve_FromData': 'Data-driven fit; sensitive when tail history is sparse.',
            'MatureReportedPaidProxy': 'Good if paid-vs-reported have stable relationship at mature ages.',
        },
    }
)
tail_candidates


## Steps 6-7: Selected CDFs, Ultimate, and IBNR

First, compare total ultimate/IBNR sensitivity across tail choices.
Then select one tail assumption for the final AY-level projection table.


In [ ]:
sensitivity_rows = []
for label, row in tail_candidates.iterrows():
    tail_factor = float(row['TailFactor'])
    tailed = cl.TailConstant(tail=tail_factor).fit_transform(selected_dev_triangle)
    model = cl.Chainladder().fit(tailed)
    sensitivity_rows.append(
        {
            'TailOption': label,
            'TailFactorUsed': tail_factor,
            'TotalLatest': float(tailed.latest_diagonal.sum()),
            'TotalUltimate': float(model.ultimate_.sum()),
            'TotalIBNR': float(model.ibnr_.sum()),
        }
    )

tail_sensitivity = pd.DataFrame(sensitivity_rows).set_index('TailOption').sort_values('TotalUltimate')

tail_sensitivity


In [ ]:
selected_tail_label = 'IndustryBenchmark_example' # choose based on considerations and sensitivity results
selected_tail_factor = float(tail_candidates.loc[selected_tail_label, 'TailFactor'])

selected_tail_model = cl.TailConstant(tail=selected_tail_factor).fit(selected_dev_triangle)
selected_tail_model

In [ ]:
selected_tail_model.cdf_

### How CDFs Are Built from Selected LDFs (Step 6)

The CDF for any development age is the **right-to-left cumulative product** of all future LDFs from that age onward, including the tail:

$$\text{CDF}(d \to \text{ult}) = \underbrace{\text{LDF}(d \to d{+}1)}_{\text{next age}} \times \underbrace{\text{LDF}(d{+}1 \to d{+}2)}_{\cdots} \times \cdots \times \underbrace{\text{Tail}}_{\text{beyond last age}}$$

The table below shows the chain explicitly — each row is a development age, and the CDF in that row is the product of all LDFs from that age through ultimate.

> **Exam tip** — `% paid = 1/CDF` is the single most important derived quantity. It tells you how developed each accident year currently is and it appears in the BF formula: `IBNR = Expected Claims × % Unreported = Expected Claims × (1 − 1/CDF)`. A recent AY with a large CDF (small `% paid`) is highly leveraged — small changes in LDF selection produce large changes in IBNR.

In [ ]:
# Build the LDF→CDF chain explicitly to show the Step 6 multiplication.
# selected_ldf is the hybrid LDF series from the candidate comparison above.
# selected_tail_factor is the chosen tail from Step 5.

ldf_values  = selected_ldf.values.tolist()
transitions = selected_ldf.index.tolist()      # e.g. ['12-24', '24-36', ...]
start_ages  = [t.split('-')[0] for t in transitions]
end_ages    = [t.split('-')[1] for t in transitions]

# Right-to-left cumulative product: CDF(d) = LDF(d) × CDF(d+1)
all_ldfs   = ldf_values + [selected_tail_factor]
cdf_values = []
running    = 1.0
for ldf in reversed(all_ldfs):
    running *= ldf
    cdf_values.insert(0, running)

# Label rows: LDF rows align to transitions; the tail gets its own row
row_labels = transitions + [f"{end_ages[-1]}-ult (Tail)"]

chain_df = pd.DataFrame({
    'Selected LDF': all_ldfs,
    'CDF to Ultimate': cdf_values,
    '% Paid  (= 1/CDF)': [1.0 / c for c in cdf_values],
    '% Unreported (= 1 − 1/CDF)': [1.0 - 1.0 / c for c in cdf_values],
}, index=pd.Index(row_labels, name='Age Transition'))

chain_df.style.format({
    'Selected LDF': '{:.4f}',
    'CDF to Ultimate': '{:.4f}',
    '% Paid  (= 1/CDF)': '{:.1%}',
    '% Unreported (= 1 − 1/CDF)': '{:.1%}',
})

**Justification:**  
* Late-age development data may be sparse and less credible.  
* External benchmarks provide stability when internal late emergence is limited.  
* The selected benchmark is consistent with plausible long-tail behavior.  
* Sensitivity analysis shows the impact is reasonable and not extreme.  

***Alternative justifications*** (if needed):  
* If late data were stable and credible, TailCurve could be defended.
* If business appeared fully developed, No Tail could be justified.
* If reported/paid maturity relationship is stable, proxy approach may be preferred.

## Final Projection and Actuarial Interpretation

Key assumptions (development method):
1. Future development follows patterns reasonably consistent with observed history.
2. Immature AY observed claims contain information about unobserved emergence.
3. Processes, mix, and program terms are stable enough for pattern transferability.

Method tends to work best with credible historical volume and relatively stable reporting/settlement behavior.


In [ ]:
selected_tailed_triangle = selected_tail_model.transform(selected_dev_triangle)
selected_model = cl.Chainladder().fit(selected_tailed_triangle)
selected_model

In [ ]:
latest = selected_tailed_triangle.latest_diagonal.to_frame().copy()
latest.columns = ['Latest']
ultimate = selected_model.ultimate_.to_frame().copy()
ultimate.columns = ['Ultimate']
ibnr = selected_model.ibnr_.to_frame().copy()
ibnr.columns = ['IBNR']

ay_projection = latest.join(ultimate).join(ibnr)
ay_projection.index = ay_projection.index.year
ay_projection.index.name = 'AccidentYear'
ay_projection.loc['Total'] = ay_projection.sum()
ay_projection

**Extended projection table — % paid and % unreported by accident year**

`% Paid = Latest / Ultimate = 1/CDF` at the AY's current development age.  
`% Unreported = 1 − % Paid`.

These columns connect the chain ladder output directly to the formula sheet and to the BF method (which uses `% Unreported` to weight between experience and a priori).  The most recent accident year has the lowest `% Paid`, meaning it is most leveraged to LDF selection errors.

In [ ]:
# Extended AY table: add CDF, % paid, and % unreported derived directly from the projection
ay_ext = ay_projection.loc[ay_projection.index != 'Total'].copy()
ay_ext['CDF_to_ult']    = ay_ext['Ultimate'] / ay_ext['Latest']
ay_ext['pct_paid']      = ay_ext['Latest']   / ay_ext['Ultimate']
ay_ext['pct_unreported']= 1.0 - ay_ext['pct_paid']

# Totals row (CDF/% columns are weighted averages, not simple sums)
totals = ay_ext[['Latest', 'Ultimate', 'IBNR']].sum()
totals['CDF_to_ult']     = totals['Ultimate'] / totals['Latest']
totals['pct_paid']       = totals['Latest']   / totals['Ultimate']
totals['pct_unreported'] = 1.0 - totals['pct_paid']
ay_ext.loc['Total'] = totals

ay_ext.style.format({
    'Latest':         '{:,.0f}',
    'Ultimate':       '{:,.0f}',
    'IBNR':           '{:,.0f}',
    'CDF_to_ult':     '{:.4f}',
    'pct_paid':       '{:.1%}',
    'pct_unreported': '{:.1%}',
})

In [ ]:
impact_table = pd.DataFrame(
    [
        ['Increase in exposure', 'Claims increase; factors broadly unchanged if average accident date is unchanged', 'Claims increase; factors broadly unchanged if average accident date is unchanged'],
        ['Average accident date shifts forward', 'Development factors and ultimates tend to be understated', 'Development factors and ultimates tend to be understated'],
        ['Increase claim ratios', 'Claims increase, development factors often less affected', 'Claims increase, development factors often less affected'],
        ['Speedup in settlement rate', 'Can overstate paid-based development and ultimate', 'Often limited direct effect'],
        ['Increase in case outstanding adequacy', 'Often limited direct effect', 'Can overstate reported-based development and ultimate'],
        ['Change in product mix', 'Patterns and selected factors should be re-segmented/reselected', 'Patterns and selected factors should be re-segmented/reselected'],
    ],
    columns=['Description', 'Paid impact', 'Reported impact'],
)
impact_table


### Development Method Leverage — When the Method Is Unreliable

The extended projection table above shows that the most recent accident year has the highest CDF and the lowest `% paid`.  This is called **method leverage**: any error in the selected LDFs is multiplied by the full remaining CDF, so small selection differences produce large ultimate differences for immature years.

**Worked example of leverage:**  
If the most recent AY has `% paid = 20%` (CDF = 5.0×), a selected LDF that is 2% too high increases the projected ultimate by roughly 2% × 5.0 = ~10%.  For a more mature AY with `% paid = 80%` (CDF = 1.25×), the same 2% LDF error produces only ~2.5% ultimate error.

**What this means for method selection:**
- For **mature accident years** (high `% paid`, CDF close to 1.0), the development method is stable and reliable.
- For **immature accident years** (low `% paid`, high CDF), the BF method is preferred because it anchors part of the estimate to an *a priori* expected loss rather than relying entirely on the leveraged development factor.
- The blend between the two is exactly the BF credibility formula: `Ultimate = (1/CDF) × Chainladder + (1 − 1/CDF) × Expected`.

## Systematic Sensitivity with `Pipeline` and `GridSearch`

Earlier in Steps 3–4 we built four candidate LDF sets by hand and compared them in a table. `chainladder` provides two tools that automate this completely:

- **`Pipeline`** — chains `Development → Tail → Chainladder` into a single sklearn-style estimator. Every parameter is addressed with `step__param` notation, so swapping assumptions requires changing one argument instead of rewriting three calls.

- **`GridSearch`** — fits every combination in a parameter grid and records the output of a scoring function. The `results_` table makes it easy to see how IBNR changes across the whole assumption space at once.

The two are most useful together: `GridSearch(Pipeline(...), param_grid={...})` is the canonical way to run a systematic sensitivity analysis in chainladder.

> **Exam relevance** — Exam 5 questions often ask how a change in assumption (e.g., more recent weighting, different tail) affects the indicated reserve. `GridSearch` answers that question for *all* combinations simultaneously, which is useful for building intuition before an exam and for explaining reserve ranges in practice.

In [ ]:
pipe.named_steps

In [ ]:
# Pipeline: replaces the manual Development() → TailCurve() → Chainladder() chain.
#
# Each step is named ('dev', 'tail', 'model') and its parameters are accessed
# with the step__param convention used by GridSearch below.
pipe = cl.Pipeline([
    ('dev',  cl.Development()),
    ('tail', cl.TailCurve(curve='exponential')),
    ('model', cl.Chainladder()),
])
pipe.fit(triangle)

# option 1 – use the pipeline proxy exactly as before
for step_name in ['dev', 'tail', 'model']:
    print(f"Step '{step_name}' parameters:")
    step = pipe.named_steps[step_name]
    for param, value in step.get_params().items():
        print(f"  {param}: {value}")
    print()
pipe_ibnr_total = pipe.named_steps.model.ibnr_.sum('origin')
pipe_ibnr_total

In [ ]:
# GridSearch: fit every combination of parameters and score each one.
#
# param_grid keys follow the step__param naming convention:
#   dev__average    — LDF averaging method ('volume', 'simple', 'regression')
#   dev__n_periods  — number of periods used in averaging (-1 = all)
#   tail__curve     — tail extrapolation curve ('exponential', 'inverse_power')
#
# scoring receives the fitted pipeline and returns a scalar that GridSearch records.
# Here we score on total IBNR, but you could score on any output (e.g., a specific AY).

pipe = cl.Pipeline(
    steps=[('dev', cl.Development()),
           ('model', cl.Chainladder())])

grid = cl.GridSearch(
    estimator=pipe,
    param_grid={'dev__average' :['volume', 'simple', 'regression']},
    scoring=lambda x : x.named_steps.model.ibnr_.sum('origin'))

grid.fit(triangle)

ax = grid.results_.set_index('dev__average').rename(columns={'score': 'IBNR'}).plot(
    kind='bar', legend=False, xlabel='', rot=0, title='GridSearch Results')

In [ ]:
# Rank results from lowest to highest IBNR to see the full reserve range.
# The spread across this table is your model risk for these assumptions.

results = (
    grid.results_
    .rename(columns={'score': 'total_ibnr'})
    .reset_index(drop=True)
)

results

---
# Exam 5 Practice Problems — Chain Ladder Hand Calculations

The following problems mirror Exam 5 written question format.
Work through each calculation before checking the solution code.
All arithmetic is reproducible without code — that is how the exam works.

**Instructions:** Show all work. Round factors to 3 decimal places. Round dollars to nearest whole number.


## Practice Problem 1: Full Chain Ladder Projection

**Exam-style question:** “Given the following triangle of cumulative paid claims (000s), use the volume-weighted average development method to project ultimates and IBNR for all accident years. Assume no tail factor.”

| AY   |  12   |  24   |  36   |  48   |
|------|------:|------:|------:|------:|
| 2020 | 1,000 | 1,500 | 1,800 | 1,872 |
| 2021 | 1,200 | 1,800 | 2,160 |       |
| 2022 | 1,400 | 2,100 |       |       |
| 2023 | 1,600 |       |       |       |

**(a)** Compute all age-to-age link ratios for each development period.  
**(b)** Select volume-weighted average LDFs.  
**(c)** Calculate CDFs to ultimate (right-to-left cumulative product).  
**(d)** Project ultimate claims and IBNR for each accident year.  


In [ ]:
import numpy as np
import pandas as pd

# PRACTICE PROBLEM 1 SOLUTION
# Cumulative paid claims triangle (000s)
raw = {
    'AY': [2020, 2021, 2022, 2023],
    12:   [1000, 1200, 1400, 1600],
    24:   [1500, 1800, 2100, None],
    36:   [1800, 2160, None, None],
    48:   [1872, None, None, None],
}
df = pd.DataFrame(raw).set_index('AY')
print('=== TRIANGLE (cumulative paid, 000s) ===')
print(df.to_string())
print()

# STEP 2: Individual link ratios
print('=== STEP 2: AGE-TO-AGE LINK RATIOS ===')
pairs = [(12, 24), (24, 36), (36, 48)]
link_ratios = {}
for (a, b) in pairs:
    ratios = [(ay, df.loc[ay, b] / df.loc[ay, a])
              for ay in df.index
              if pd.notna(df.loc[ay, a]) and pd.notna(df.loc[ay, b])]
    link_ratios[(a, b)] = ratios
    print(f'  {a}-{b}:  ' + '  '.join(f'AY{ay}: {r:.3f}' for ay, r in ratios))
print()

# STEP 3: Volume-weighted averages
print('=== STEP 3: VOLUME-WEIGHTED AVERAGE LDFs ===')
selected_ldfs = {}
for (a, b) in pairs:
    num = sum(df.loc[ay, b] for ay, _ in link_ratios[(a, b)])
    den = sum(df.loc[ay, a] for ay, _ in link_ratios[(a, b)])
    vw  = num / den
    selected_ldfs[(a, b)] = vw
    print(f'  LDF {a}-{b}:  {num:,.0f} / {den:,.0f} = {vw:.3f}')
print()

# STEP 6: CDFs right-to-left (no tail so CDF at 48 = 1.000)
print('=== STEP 6: CUMULATIVE DEVELOPMENT FACTORS (to ultimate) ===')
cdf = {48: 1.000}
cdf[36] = selected_ldfs[(36, 48)] * cdf[48]
cdf[24] = selected_ldfs[(24, 36)] * cdf[36]
cdf[12] = selected_ldfs[(12, 24)] * cdf[24]
for age in [12, 24, 36, 48]:
    print(f'  CDF({age}->ult) = {cdf[age]:.3f}   [% paid = {1/cdf[age]*100:.1f}%]')
print()

# STEP 7: Ultimates and IBNR
print('=== STEP 7: ULTIMATE AND IBNR ===')
latest_age = {2020: 48, 2021: 36, 2022: 24, 2023: 12}
rows = []
for ay in df.index:
    age    = latest_age[ay]
    latest = df.loc[ay, age]
    ult    = latest * cdf[age]
    ibnr   = ult - latest
    rows.append({'AY': ay, 'Age': age, 'Latest Paid': latest,
                 'CDF': round(cdf[age], 3), 'Ultimate': round(ult, 1), 'IBNR': round(ibnr, 1)})

res = pd.DataFrame(rows).set_index('AY')
res.loc['Total'] = ['', res['Latest Paid'].sum(), '', res['Ultimate'].sum(), res['IBNR'].sum()]
print(res.to_string())
print()
print('KEY CHECKS:')
print('  IBNR = Ultimate - Latest Paid  (never skip this subtraction!)')
print(f'  AY 2020 fully developed -> CDF = 1.000, IBNR = 0')
print(f'  AY 2023 most leveraged  -> CDF = {cdf[12]:.3f}, % paid = {1/cdf[12]*100:.1f}%')
print('  If asked for IBNR by accident year, give a table -- not just a total.')


## Practice Problem 2: Tail Factor Application and Sensitivity

**Exam-style question:** “An actuary has completed the development projection through 60 months. Selected LDFs and the latest diagonal values are shown below.”

| Period | Selected LDF |
|--------|-------------:|
| 12–24  | 2.100        |
| 24–36  | 1.400        |
| 36–48  | 1.150        |
| 48–60  | 1.060        |
| 60+    | ?            |

| AY   | Latest (000s) | Age   |
|------|-------------:|-------|
| 2020 | 4,800        | 60 mo |
| 2021 | 4,200        | 48 mo |
| 2022 | 3,100        | 36 mo |
| 2023 | 2,400        | 24 mo |
| 2024 | 1,500        | 12 mo |

**(a)** Using a tail of **1.050**, calculate the CDF for each age and project total IBNR.  
**(b)** A colleague argues the tail should be **1.000**. Calculate total IBNR under this assumption.  
**(c)** Briefly explain which tail is more defensible and why. (3–4 sentences max)  


In [ ]:
import pandas as pd
import numpy as np

# PRACTICE PROBLEM 2 SOLUTION
ldf_seq = [2.100, 1.400, 1.150, 1.060]   # 12->24, 24->36, 36->48, 48->60
age_seq = [12, 24, 36, 48, 60]

latest_data = {
    2020: (4800, 60),
    2021: (4200, 48),
    2022: (3100, 36),
    2023: (2400, 24),
    2024: (1500, 12),
}

def build_cdfs(tail):
    # CDF at each age = product of all future LDFs right-to-left including tail
    full_ldfs = ldf_seq + [tail]
    full_ages = age_seq + [9999]   # 9999 = beyond tail
    cdf = {9999: 1.000}
    for i in range(len(full_ldfs) - 1, -1, -1):
        cdf[full_ages[i]] = full_ldfs[i] * cdf[full_ages[i + 1]]
    return cdf

def project(tail, label):
    cdf = build_cdfs(tail)
    print(f'\n{"="*58}')
    print(f'  Tail = {tail:.3f}  --  {label}')
    print(f'{"="*58}')
    print(f'  {"Age":>4}  {"CDF":>7}  {"% Paid":>8}  {"% Unreported":>13}')
    for age in age_seq:
        pct = 1 / cdf[age]
        print(f'  {age:>4}  {cdf[age]:>7.3f}  {pct*100:>7.1f}%  {(1-pct)*100:>12.1f}%')
    print()
    rows = []
    for ay, (latest, age) in latest_data.items():
        ult  = latest * cdf[age]
        ibnr = ult - latest
        rows.append({'AY': ay, 'Age': age, 'Latest': latest,
                     'CDF': round(cdf[age], 3), 'Ultimate': round(ult), 'IBNR': round(ibnr)})
    res = pd.DataFrame(rows).set_index('AY')
    res.loc['Total'] = ['', '', res['Latest'].sum(), '', res['Ultimate'].sum(), res['IBNR'].sum()]
    print(res.to_string())
    return res.loc['Total', 'IBNR']

ibnr_a = project(1.050, 'Industry benchmark tail')
ibnr_b = project(1.000, 'No tail -- fully developed assumption')

diff = ibnr_a - ibnr_b
print(f'\n>>> IBNR difference from tail assumption: ${diff:,.0f} (000s)')
print(f'>>> Tail adds {diff/ibnr_b*100:.1f}% to IBNR -- material for a long-tail line')
print()
print('PART (c) -- MODEL ANSWER (written exam format):')
print('-' * 58)
print(
    'A tail factor of 1.000 assumes claims are fully developed at 60 months,\n'
    'which is generally not appropriate for auto liability where late-reported\n'
    'claims and reopened cases can emerge beyond that age.\n'
    '\n'
    'The 1.050 benchmark is more defensible because it reflects residual\n'
    'development consistent with industry experience for medium-tail lines.\n'
    'Unless the actuary has specific data confirming complete development at\n'
    '60 months, 1.000 likely understates reserves.\n'
    '\n'
    'COMMON TRAP: Forgetting to multiply the tail into the CDF chain is the\n'
    'most penalized mechanical error on tail questions.'
)


## Practice Problem 3: Paid vs. Incurred — Comparison and Judgment

**Exam-style question:** “An actuary observes the following volume-weighted LDFs:”

| Period | Paid LDF | Reported LDF |
|--------|:--------:|:------------:|
| 12–24  |  2.350   |    1.980     |
| 24–36  |  1.420   |    1.310     |
| 36–48  |  1.180   |    1.100     |
| 48–60  |  1.050   |    1.030     |
| 60–72  |  1.020   |    1.010     |
| Tail   |  1.015   |    1.005     |

Projected ultimates: **Paid = $42,500** (000s) | **Reported = $39,200** (000s)

**(a)** Calculate the total CDF to ultimate for each method.  
**(b)** The paid ultimate is $3,300 higher. Give **two** explanations. For each, state whether it implies the Paid or Reported method is more reliable.  
**(c)** In 2–3 sentences: when would you prefer the Reported method over Paid?  


In [ ]:
import numpy as np

# PRACTICE PROBLEM 3 SOLUTION
periods       = ['12-24', '24-36', '36-48', '48-60', '60-72', 'Tail']
paid_ldfs     = [2.350,   1.420,   1.180,   1.050,   1.020,   1.015]
reported_ldfs = [1.980,   1.310,   1.100,   1.030,   1.010,   1.005]

paid_cdf     = np.prod(paid_ldfs)
reported_cdf = np.prod(reported_ldfs)

print('=== PART (a): CDF TO ULTIMATE ===')
print(f'  Paid CDF     = {" x ".join(str(f) for f in paid_ldfs)}')
print(f'               = {paid_cdf:.3f}')
print()
print(f'  Reported CDF = {" x ".join(str(f) for f in reported_ldfs)}')
print(f'               = {reported_cdf:.3f}')
print()
print(f'  Ratio (Paid/Reported): {paid_cdf/reported_cdf:.3f}')
print()

print('=== LDF COMPARISON BY PERIOD ===')
print(f'  {"Period":>7}  {"Paid":>7}  {"Reported":>9}  {"Ratio P/R":>10}')
for lbl, p, r in zip(periods, paid_ldfs, reported_ldfs):
    print(f'  {lbl:>7}  {p:>7.3f}  {r:>9.3f}  {p/r:>10.3f}')
print()

print('=== PART (b): TWO EXPLANATIONS ===')
print(
    'Explanation 1 -- Case Reserve Strengthening (favors Paid method):\n'
    '  If the company strengthened case reserves, reported claims are\n'
    '  higher than true loss emergence warrants. The recent diagonal is\n'
    '  inflated, so future increments look smaller -- reported LDFs are\n'
    '  suppressed and reported ultimates are understated.\n'
    '  --> PAID method is more reliable: unaffected by reserve changes.\n'
    '\n'
    'Explanation 2 -- Slowdown in Claims Settlement (favors Reported method):\n'
    '  If the company slowed claim payments, paid claims at each age are\n'
    '  below historical norms, causing paid LDFs to appear higher than\n'
    '  they should be going forward.\n'
    '  --> REPORTED method is more reliable: it captures case reserves\n'
    '    and is not distorted by settlement speed changes.\n'
    '\n'
    'NOTE: These two explanations lead to OPPOSITE conclusions.\n'
    'The exam may give additional data to help you determine which applies.'
)
print()
print('=== PART (c): WHEN TO PREFER REPORTED METHOD ===')
print(
    'MODEL ANSWER:\n'
    'The Reported (Incurred) method is preferred when case reserves are\n'
    'set consistently and adequately -- the incurred triangle already\n'
    'incorporates pending reserves, making it more informative for\n'
    'immature accident years where few claims have been paid.\n'
    '\n'
    'However, if case reserve adequacy has changed (strengthening or\n'
    'weakening), reported LDFs are distorted and the Paid method is\n'
    'more objective.\n'
    '\n'
    'COMMON TRAP: Never say reported is always better for immature years\n'
    'without first confirming that case reserves have been stable.'
)


---
## When Chain Ladder Is Inappropriate — Complete Reference

Pure conceptual question type. Know the full list, the direction of bias, and what method to use instead.

### Chain Ladder Assumptions (must all hold):
1. Future development **follows observed historical patterns**
2. Data is **homogeneous** — same mix of business, same handling procedures
3. No **material operational changes** during the experience period
4. **Adequate and consistent case reserving** (for reported/incurred triangle)
5. No **structural changes** to the book of business

### Situations Where Chain Ladder Breaks Down:

| Situation | Why It Fails | Direction of Bias |
|---|---|:---:|
| **Rapid premium growth** | Recent AYs are thin; small dollar errors cause large % errors in leveraged CDFs | Unreliable |
| **Claim handling changes** | Settlement speed shifts distort LDFs; prior patterns don’t transfer | Either |
| **Case reserve strengthening** | Inflates reported triangle; reported LDFs appear lower than true | Understates reported ult |
| **Case reserve weakening** | Deflates reported triangle; reported LDFs appear higher | Overstates reported ult |
| **Inflation shock** | Calendar year diagonal effect; LDFs not stable across AYs | Understates if inflation accelerated |
| **Legislative reform** | Changes claim frequency, severity, or tail; prior patterns invalid | Either |
| **Catastrophe distortion** | Single large event inflates one diagonal; not representative of normal development | Overstates if included |
| **Line of business mix shift** | Prior patterns reflect old mix; new mix has different development | Either |
| **Very immature data** | CDF heavily leveraged; small errors magnified | High variance |

### Exam-Ready Answer Template:
> “Chain Ladder is inappropriate here because [specific situation] causes the historical
> development patterns to be **not representative** of future development. Specifically,
> [mechanism]. This would cause the method to **[overstate / understate]** IBNR.
> A preferable alternative is [BF / Cape Cod / judgment-adjusted] because [reason].”

### When to Use BF Instead of Chain Ladder:
- Accident years where **% paid < 30%** — too leveraged on sparse early data
- After a **shock loss** distorts recent diagonals
- When **a priori expected loss ratios** are credible (new line, startup book)
- When **data volume is thin** and statistical credibility is low

BF blend: `Ultimate = Latest + (1 − 1/CDF) × Expected`  
At low % paid, BF is mostly driven by expected; at high % paid, BF converges to Chain Ladder.


---
## Exam-Style Written Answer Examples — Judgment Questions

### How Written Answers Are Scored

Examiners score on **content points**, not length. A 3-sentence answer that hits all key points outscores a paragraph that restates numbers.

> **Examiner note (Fall 2018):** *“Numerical restatement is not justification. Candidates who simply listed the selected factors received no credit for the justification portion.”*

---

### Example A — Factor Selection Justification

**Question:** “The 12–24 age-to-age factors show a downward trend from 1.85 (AY 2016) to 1.52 (AY 2022). Briefly explain how this affects your LDF selection.” *(3 points)*

**Weak answer (1/3 points):**
> “The factors decreased over time so I would select a lower factor.”

**Strong answer (3/3 points):**
> “The downward trend in 12–24 factors may indicate faster early reporting or improved claims handling in recent years. If this trend reflects a structural change rather than random fluctuation, I would give more weight to recent accident years — for example, using a 5-year volume-weighted average rather than an all-year average. Using older, higher factors would overstate IBNR for the most recent, most leveraged accident years.”

**Why it works:** Names a mechanism, states a selection action, and identifies the direction of impact.

---

### Example B — Excluding Accident Years

**Question:** “Explain two reasons you might exclude the most recent accident year from your LDF selection.” *(4 points)*

**Strong answer:**
> “First, the most recent accident year is at its earliest development age, where very few claims have been reported. The factor is highly volatile and statistically unreliable, adding noise without meaningful credibility to the average.
>
> Second, for a long-tail line, the 12-month evaluation may reflect only fast-settling claims, which are not representative of the ultimate settlement pattern — including them would bias the factor downward and understate development.”

---

### Example C — Calendar Year Diagonal Effect

**Question:** “A calendar year diagonal shows significantly above-average development across all accident years. Identify two possible causes and explain the implication for Chain Ladder.” *(4 points)*

**Strong answer:**
> “Two possible causes: (1) **Social inflation** — a spike in claim severity or jury awards in that calendar year would appear as above-average payments across multiple accident years simultaneously, creating a diagonal pattern. (2) **Claims settlement acceleration** — if the company expedited payments in that year, more losses would be recorded across the diagonal.
>
> The implication for Chain Ladder is that these diagonal effects violate the stability assumption — the method assumes development depends only on accident year age, not on which calendar year the payments occur. Including this distorted diagonal in LDF averaging will overstate future development and overstate IBNR.”

---

### Example D — Paid vs. Incurred Divergence

**Question:** “The paid ultimate is $3.2M higher than the incurred ultimate. Give one explanation and state which method you would rely on.” *(3 points)*

**Strong answer:**
> “One explanation is case reserve strengthening: if the company increased case reserves in recent years, reported claims at early ages are higher than historical norms. This suppresses reported LDFs (less incremental growth needed), causing the reported method to understate the ultimate.
>
> I would rely on the **paid method** in this situation because paid claims are unaffected by case reserve changes and reflect actual cash outflows.”

---

### Key Phrases Examiners Reward:
- *“…violates the stability assumption…”*
- *“…direction of impact is [overstates / understates] IBNR because…”*
- *“…most leveraged for immature accident years…”*
- *“…not representative of future patterns because [specific mechanism]…”*
- *“…I would weight recent periods more heavily / exclude older years because…”*


---
## Data Organization: Calendar Year vs. Accident Year vs. Policy Year

Lower-frequency topic but tested via CAS Ratemaking and Reserving principles.

| Basis | Definition | Best For | Key Drawback |
|---|---|---|---|
| **Accident Year (AY)** | Groups losses by when the accident occurred | Loss reserving; matches emergence to exposure period | Requires IBNR estimates for late-reported claims |
| **Calendar Year (CY)** | Groups losses by when recorded/paid in the accounting period | Financial statements; simplest to compile | Mixes accident years; useless for development analysis |
| **Policy Year (PY)** | Groups losses by when the policy was written | Ratemaking; cleanest match of losses to rate level | Slowest to mature (wait for all policies to expire and fully develop) |
| **Report Year** | Groups losses by when first reported | IBNR analysis for claims-made policies | Less intuitive for occurrence-based lines |

### Exam-Ready Comparison:
- **Accident Year** is the standard for **loss reserving** (Chain Ladder, BF, Cape Cod)
- **Policy Year** is preferred for **ratemaking** when matching losses to specific rate levels — but takes the longest to develop
- **Calendar Year** is the **accounting view** — do not use for development analysis; the diagonal of a Chain Ladder triangle is a calendar year valuation, not an axis

### Common Trap:
Chain Ladder triangles are organized by **Accident Year × Development Age**. The *diagonal* of the triangle represents a single **calendar year** (valuation date). Mixing these up — treating a diagonal trend as an accident year pattern — is a common exam error.

> If you see above-average factors along a **diagonal**: that is a **calendar year effect**.  
> If you see above-average factors in a **row**: that is an **accident year effect**.


---
## Exam Trap Awareness — Most Common Point-Losing Mistakes

From CAS Examiner Reports (Fall 2018–2019) and general exam patterns.

### Mechanical Traps

| Trap | What Goes Wrong | How to Avoid |
|---|---|---|
| **Wrong average method** | Using simple average when volume-weighted is required | Default to volume-weighted unless stated otherwise |
| **Forgetting IBNR = Ultimate − Reported** | Reporting the Ultimate as the IBNR answer | Always subtract the latest diagonal |
| **Omitting tail factor** | CDF chain doesn’t include tail; ultimate understated | Always ask: is this fully developed? If not, apply a tail |
| **CDF arithmetic error** | Dividing instead of multiplying; wrong order | CDF > 1.0 always; CDF(12) = LDF(12-24) × LDF(24-36) × … × Tail |
| **Mixing paid and incurred** | Applying paid LDFs to incurred triangle | Label every triangle; keep methods completely separate |
| **Wrong multiplication order** | Left-to-right instead of right-to-left | Build CDF from the oldest age back; multiply leftward |

### Judgment / Written Answer Traps

| Trap | What Goes Wrong | How to Avoid |
|---|---|---|
| **Restating numbers** | “The 12–24 factor is 1.45” — this describes, it does not justify | Always follow with “this suggests…” or “because…” |
| **No direction of impact** | Saying an assumption is violated without stating which way | Always state: this would overstate / understate IBNR because… |
| **Generic answers** | “Factors were volatile” with no specifics | Name the distortion, name the period, name the corrective action |
| **Ignoring ‘briefly’** | Writing 10 sentences when asked to briefly explain | ‘Briefly’ = 2–3 sentences; verbosity is penalized |
| **Confusing AY and CY effects** | Treating a diagonal trend as an accident year phenomenon | AY effect = row pattern; CY effect = diagonal pattern |
| **Omitting % paid discussion** | Not noting that immature AYs are highly leveraged | Comment on % paid for the most recent AY whenever relevant |

---

### Quick Self-Check Before Finalizing Any Chain Ladder Answer:

- [ ] Did I use **volume-weighted** (not simple) average unless stated otherwise?
- [ ] Did I include a **tail factor** and multiply it into the CDF chain?
- [ ] Did I compute **IBNR = Ultimate − Latest Paid** (not just state the Ultimate)?
- [ ] Did I **justify** my selections (not just restate numbers)?
- [ ] Did I identify the **direction of bias** (overstate vs. understate)?
- [ ] If calendar year effects — did I discuss **diagonals**, not rows?
- [ ] If comparing paid vs. incurred — did I discuss **case reserve stability**?
- [ ] For ‘briefly’ questions — did I keep my answer to **2–3 sentences**?
